---
## Section 5 — LLM Answer Generation

We now combine retrieval with an LLM to get **grounded answers** — answers that are based on actual documents, with source URLs you can verify.

### Three retrieval modes

| Mode | What it does |
|------|--------------|
| `"hybrid"` | Queries **all three** sources, merges and deduplicates results, feeds everything to the LLM |
| `"specific"` | Uses only the one source you choose: `local`, `remote`, or `ourss` |
| `"agentic"` | The LLM itself decides which source(s) to call, then synthesises an answer |

**Change `MODE` and `YOUR_QUESTION` below**, then run all cells in order.

In [1]:
# ── 5.0  Configuration — change these 
MODE          = "hybrid"   # "hybrid" | "specific" | "agentic"
SOURCE        = "remote"   # used only when MODE == "specific"
TOP_N         = 10

YOUR_QUESTION = "What is the Open Web Index and how can researchers access it?"

In [2]:
# ── 5.1  Unified retrieval dispatcher 

from m1.retriever import retrieve_local, retrieve_ourss, retrieve_remote
from m1.utils.mosaic_tools import _dedup


def retrieve(queries, n: int = TOP_N,
             mode: str = MODE, source: str = SOURCE) -> list:
    """Route a query to the correct retrieval function(s) based on mode."""
    if mode == "hybrid":
        combined = (retrieve_local(queries, n)
                    + retrieve_remote(queries, n)
                    + retrieve_ourss(queries, n))
        return _dedup(combined)
    elif mode == "specific":
        fn = {"local": retrieve_local,
              "remote": retrieve_remote,
              "ourss":  retrieve_ourss}.get(source)
        assert fn, f"Unknown source '{source}'. Choose: local, remote, ourss"
        return fn(queries, n)
    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'hybrid', 'specific', or 'agentic'.")

print("Retrieval dispatcher ready ✅")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Retrieval dispatcher ready ✅


In [3]:
# ── 5.2  LangChain RAG chain 
# LLM is configurable — change this line to switch between small and large.
# small: faster and cheaper   |   large: better reasoning, higher quality answers
import os

from langchain.messages import HumanMessage, SystemMessage
from langchain_mistralai import ChatMistralAI


LLM_API_KEY    = os.getenv("LLM_API_KEY")
LLM_MODEL = "mistral-small-2603"

LLM = ChatMistralAI(
    api_key=LLM_API_KEY,
    model=LLM_MODEL,
    temperature=0.7,
)


## PROMPTS - FEEL FREE TO CHANGE
sys_prompt = """
You are an academic research assistant helping users understand topics based on web search results.
"""

RAG_PROMPT = """

# TASK:
- Answer the user's question using the information provided in the retrieved context below.

Citation rules:
- Cite sources using [Source N] notation, where N refers to the numbered context passages.

Format:
- Structure longer answers with clear paragraphs or bullet points where helpful.
- End with a "Sources" section listing the numbered sources you cited, with their URLs.

Context:
{context}

Question:
{question}
"""

def format_context(results: list) -> str:
    """Convert a list of result dicts into a numbered context string."""
    parts = []
    for i, r in enumerate(results, 1):
        parts.append(
            f"[{i}] Title: {r.get('title','')}\n"
            f"URL: {r.get('url','')}\n"
            f"{r.get('text','')}"
        )
    return "\n\n".join(parts)


In [4]:
# ── 5.3  Hybrid / Specific mode — answer a question 

from m1.utils.mosaic_tools import display_results


if MODE in ("hybrid", "specific"):
    label = MODE + (f" [{SOURCE}]" if MODE == "specific" else "")
    print(f"Mode: {label}")
    print(f"Question: {YOUR_QUESTION}\n")

    context_results = retrieve(YOUR_QUESTION)
    display_results(context_results, "Retrieved context passages")

    context_str = format_context(context_results)
    answer = LLM.invoke([
            SystemMessage(content=sys_prompt),
            HumanMessage(content=RAG_PROMPT.format(context=context_str, question=YOUR_QUESTION))
        ])


    print("\n" + "═"*60)
    print("  ANSWER")
    print("═"*60)
    print(answer.content)

Mode: hybrid
Question: What is the Open Web Index and how can researchers access it?




════════════════════════════════════════════════════════════
  ANSWER
════════════════════════════════════════════════════════════
The **Open Web Index** is a concept that has been discussed in various contexts, but it isn't a universally standardized or officially recognized term. However, based on the provided context passages, it appears to be related to open access, digital repositories, and alternative metrics for scholarly communication. Here’s what can be inferred from the context:

1. **Open Web Index as a Repository or Indexing Service**:
   - The term may refer to an **open repository** or **indexing service** that aggregates and makes scholarly content freely accessible.
   - For example, platforms like **OpenEdition Books** (mentioned in [Source 20](https://www.ingentaconnect.com/about/help/index;jsessionid=5q23e7iv9vbv4.x-ic-live-03)) or **OpenUCT** (discussed in [Source 29](https://www


In [5]:
# ── 5.4  Agentic mode — let the LLM choose which sources to query 
#
# The LLM is given three tools (one per retrieval source) and autonomously
# decides which to call. The system prompt below is intentionally minimal —
# improving it is a great experiment!
#
# Uses llm_large by default because agentic reasoning benefits from the
# stronger model. Change AGENTIC_LLM to llm_small to trade quality for speed.

from langchain.messages import ToolMessage
from langchain.tools import tool
from m1.utils.display_tools import display_agentic_answer


AGENTIC_LLM = LLM   # ← swap to llm_small if needed

# Define tools using LangChain's @tool decorator - FEEL FREE TO CHANGE THE description of each
@tool
def search_local(query: str, n: int = 5) -> str:
    """Search the local MOSAIC index (offline, built from downloaded OWI data).
    Best for questions about the specific dataset you downloaded."""
    results = retrieve_local(query, n)
    return format_context(results) if results else "No results found."

@tool
def search_remote(query: str, n: int = 5) -> str:
    """Search the remote MOSAIC index provided by the organizers.
    Covers a larger and more up-to-date OWI snapshot."""
    results = retrieve_remote(query, n)
    return format_context(results) if results else "No results found."

@tool
def search_ourss(query: str, n: int = 5) -> str:
    """Search the ourrs.eu open web search API.
    Good for broad, general-purpose web queries."""
    results = retrieve_ourss(query, n)
    return format_context(results) if results else "No results found."

AGENTIC_TOOLS = [search_local, search_remote, search_ourss]
llm_with_tools = AGENTIC_LLM.bind_tools(AGENTIC_TOOLS)
tool_map = {t.name: t for t in AGENTIC_TOOLS}


def agentic_answer(question: str, max_steps: int = 4) -> str:
    """
    Agentic RAG loop: the LLM calls tools autonomously until it has enough
    context to answer the question.
    """
    messages = [
        SystemMessage(content=(
            "You are a helpful research assistant. "
            "Use the available search tools to find relevant information, "
            "then write a clear, grounded answer with source citations."
        )),
        HumanMessage(content=question),
    ]

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:  # LLM is done calling tools
            return response.content or "(no answer generated)"

        for tc in response.tool_calls:
            print(f"  Step {step+1}: calling {tc['name']}(query='{tc['args'].get('query','')}')")
            tool_result = tool_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=tool_result, tool_call_id=tc["id"]))

    return "(reached maximum steps without a final answer)"

MODE= "agentic"
if MODE == "agentic":
    print(f"Agentic mode  (model: {AGENTIC_LLM.model})")
    print(f"Question: {YOUR_QUESTION}\n")
    answer = agentic_answer(YOUR_QUESTION)
    print("\n" + "═"*60)
    print("  AGENTIC ANSWER")
    print("═"*60)
    display_agentic_answer(answer)

Agentic mode  (model: mistral-small-2603)
Question: What is the Open Web Index and how can researchers access it?

  Step 1: calling search_ourss(query='Open Web Index definition and access for researchers')
  Step 1: calling search_remote(query='Open Web Index definition and access for researchers')

════════════════════════════════════════════════════════════
  AGENTIC ANSWER
════════════════════════════════════════════════════════════
══════════════════════════════════════════════════════════════
  AGENTIC ANSWER
══════════════════════════════════════════════════════════════


The **Open Web Index (OWI)** is a comprehensive, global dataset of open access research publications, designed to facilitate text and data mining (TDM) and support scientific discovery. It aggregates metadata and full-text records from thousands of data providers worldwide, including institutional and subject repositories, journals, and databases. Below is an overview of its key features, significance, and how researchers can access it:

---

### **What is the Open Web Index?**
1. **Purpose and Scope**:
   - The OWI aims to provide a **single point of access** to the world’s open access research literature, enabling both humans and machines to discover, analyze, and reuse scientific content.
   - It addresses the challenge of fragmented and siloed research outputs by harmonizing metadata and full-text records from diverse sources.
   - The index is **continuously updated** to ensure recency and completeness, making it one of the largest and most dynamic collections of open access papers available `[Source 3]`.

2. **Key Features**:
   - **Global Coverage**: As of 2023, the OWI provides access to over **291 million metadata records** and **32.8 million full-text articles** from more than **10,000 data providers** across **150 countries** `[Source 3]`.
   - **Multilingual Support**: The dataset includes publications in **82 languages**, reflecting the global nature of research.
   - **Data Enrichment**: The OWI enriches records with additional metadata, such as language detection, document type classification (e.g., research articles, theses, presentations), and linking to external datasets like ORCID, Crossref, and PubMed.
   - **Free and Open Access**: The aggregated data is made freely available under the **ODC-By license**, allowing both commercial and non-commercial reuse with proper attribution.
   - **Multiple Access Points**: Researchers can access the data through:
     - **API**: Real-time machine access to metadata and full texts.
     - **Dataset**: Bulk downloads of the entire collection or subsets.
     - **FastSync**: A synchronization service for keeping local copies of the data up-to-date `[Source 3]`.

3. **Significance**:
   - **Scientific Discovery**: The OWI enables large-scale TDM and AI-driven analysis of research literature, supporting applications like plagiarism detection, systematic review automation, and scientific recommendation systems.
   - **Equitable Access**: By providing free access to research outputs, the OWI helps bridge the **global digital divide** in scholarly communication, particularly benefiting researchers in developing countries who may lack access to subscription-based databases.
   - **Interoperability**: The index promotes standardization and integration of research data, fostering collaboration and reducing duplication of efforts in data collection and processing `[Source 3]`.

---

### **How Can Researchers Access the Open Web Index?**
Researchers can access the OWI through several services, depending on their needs:

1. **CORE API**:
   - **Description**: The CORE API provides real-time access to the OWI’s metadata and full-text records.
   - **Access**: Users can register for an API key to query the dataset programmatically.
   - **Use Cases**: Building applications, conducting TDM, or integrating OWI data into existing workflows.
   - **Documentation**: Available at [https://api.core.ac.uk/docs/v3](https://api.core.ac.uk/docs/v3) `[Source 3]`.

2. **CORE Dataset**:
   - **Description**: The CORE Dataset allows bulk downloads of the OWI’s data in standardized formats (e.g., CSV, JSON, or XML).
   - **Access**: Researchers can register to download the dataset or subsets of it.
   - **Use Cases**: Offline analysis, large-scale data processing, or integration into local databases.
   - **License**: Data is released under the **ODC-By license**, requiring attribution for reuse `[Source 3]`.

3. **FastSync Service**:
   - **Description**: FastSync enables researchers or organizations to keep a local, always-up-to-date copy of the OWI’s data.
   - **Access**: This service is optimized for synchronization and is useful for institutions or projects that need to maintain their own mirrored versions of the dataset.
   - **Use Cases**: Enterprise-level data integration, real-time TDM, or offline research environments `[Source 3]`.

4. **CORE Discovery**:
   - **Description**: A browser extension and plugin for repositories that helps users discover open access versions of research papers.
   - **Access**: Available for integration into institutional repositories or personal workflows.
   - **Use Cases**: Finding legally accessible versions of paywalled articles or improving discoverability of open access content `[Source 3]`.

---

### **Why Use the Open Web Index?**
- **Comprehensive Coverage**: The OWI is one of the largest collections of open access research papers, significantly larger than other aggregators like **BASE** or **Unpaywall** `[Source 3]`.
- **Free and Legal Access**: Unlike some other services that only provide links to paywalled content, the OWI **hosts full-text records** and ensures they are legally available for reuse.
- **Machine-Readable Format**: The index provides plain-text versions of full texts, eliminating the need for researchers to manually convert PDFs or other formats for TDM.
- **Support for Innovation**: The OWI powers applications like **plagiarism detection (Turnitin)**, **scholarly search engines (Naver, Lean Library)**, and **recommendation systems (CORE Recommender)** `[Source 3]`.

---
### **Challenges Addressed by the OWI**
The OWI was developed to overcome several challenges in accessing and reusing open access research literature:
1. **Fragmentation**: Research papers are scattered across thousands of repositories, journals, and databases, often with incompatible metadata standards.
2. **Interoperability**: Many data providers lack support for efficient content harvesting, making it difficult to systematically gather full-text records.
3. **Scalability**: Harvesting and processing large volumes of data requires innovative solutions to manage computational resources and ensure recency.
4. **Legal and Technical Barriers**: Some repositories restrict or prohibit machine access to full texts, limiting the ability to perform TDM `[Source 3]`.

---
### **Conclusion**
The Open Web Index is a **critical infrastructure** for the global open science movement, providing researchers, institutions, and developers with free and legal access to a vast corpus of scientific literature. By enabling TDM and AI-driven analysis, the OWI supports innovation, collaboration, and equitable access to knowledge, particularly benefiting researchers in developing countries.

For more information or to access the OWI, visit [https://core.ac.uk](https://core.ac.uk) or explore its documentation and services at [https://core.ac.uk/about](https://core.ac.uk/about) `[Source 3]`.

---
## Section 6 — Evaluation (LLM-as-a-Judge)

How do we know if the answers are any good? We use a second LLM call to **score** each answer on three dimensions:

| Criterion | What it measures | Scale |
|-----------|------------------| ------|
| **Faithfulness** | Does the answer stick to the retrieved context, with no made-up facts? | 1–5 |
| **Relevance** | Does the answer directly address the question? | 1–5 |
| **Completeness** | Does the answer cover the key points available in the context? | 1–5 |

This technique is called **LLM-as-a-Judge** and is widely used in the RAG research community when human evaluation is too slow or expensive.

> **Experiment idea:** add your own scoring dimensions to `JUDGE_PROMPT_TEMPLATE` — for example *conciseness*, *citation quality*, or *tone*.

In [ ]:
# ── 6.1  Judge prompt and function 
# The judge always uses llm_large for more reliable, consistent scoring.

import json as _json

JUDGE_LLM = LLM   # ← large model gives more reliable scores

JUDGE_PROMPT = """
You are an impartial evaluation judge. Score the answer below on three criteria.
Return ONLY a JSON object — no extra text, no markdown fences.

Scoring scale: 1 = very poor, 5 = excellent

Criteria definitions:
- faithfulness  : Is every claim in the answer supported by the context? (no hallucinations)
- relevance     : Does the answer directly address the question?
- completeness  : Does the answer cover all key information available in the context?

--- QUESTION ---
{question}

--- RETRIEVED CONTEXT ---
{context}

--- ANSWER TO EVALUATE ---
{answer}

Return exactly this JSON (replace ... with actual values):
{{
  "faithfulness":  {{"score": <1-5>, "reason": "..."}},
  "relevance":     {{"score": <1-5>, "reason": "..."}},
  "completeness":  {{"score": <1-5>, "reason": "..."}}
}}
"""


def judge_answer(question: str, context: str, answer: str) -> dict:
    """Score an answer using the LangChain judge chain. Returns a dict with scores + reasons."""
    raw = answer = LLM.invoke([
            HumanMessage(content=JUDGE_PROMPT.format(question=question, context=context,answer=answer ))
        ]).content
    

    try:
        return _json.loads(raw)
    except _json.JSONDecodeError:
        print("⚠️  Could not parse judge response as JSON:\n", raw)
        return {}

print(f"Judge  ready ✅  (model: {JUDGE_LLM.model})")

Judge  ready ✅  (model: mistral-small-2603)


In [13]:
# ── 6.2  TEST — judge a known-good vs known-bad answer 

def test_judge():
    q   = "What is the Open Web Index?"
    ctx = "The Open Web Index (OWI) is a European open-source web crawl initiative."

    cases = [
        ("Faithful answer",     "The Open Web Index is a European open-source web crawl initiative."),
        ("Hallucinated answer", "The Open Web Index is a NASA satellite programme launched in 2010."),
    ]
    for label, ans in cases:
        scores = judge_answer(q, ctx, ans)
        faith  = scores.get("faithfulness", {}).get("score", "?")
        reason = scores.get("faithfulness", {}).get("reason", "")
        print(f"{label}: faithfulness = {faith}/5")
        print(f"  Reason: {reason}")
        print()
    print("✅ The faithful answer should score higher on faithfulness")

test_judge()

Faithful answer: faithfulness = 5/5
  Reason: The answer is fully supported by the context without any additions or omissions.

Hallucinated answer: faithfulness = 1/5
  Reason: The answer incorrectly attributes the Open Web Index to NASA and misstates it as a satellite programme launched in 2010, which contradicts the context describing it as a European open-source web crawl initiative.

✅ The faithful answer should score higher on faithfulness


In [ ]:
# ── 6.3  Evaluation questions — add your own! 

EVAL_QUESTIONS = [
    "What is the Open Web Index?",
    "How can researchers access European web crawl data?",
    "What is MOSAIC and how does it work?",
]

In [ ]:
# ── 6.4  Run full evaluation 
import pandas as pd

eval_rows = []

for q in EVAL_QUESTIONS:
    print(f"Evaluating: {q}")
    ctx_results = retrieve(q, n=TOP_N)
    ctx_str     = format_context(ctx_results)
    ans         =  LLM.invoke([
            SystemMessage(content=sys_prompt),
            HumanMessage(content=RAG_PROMPT.format(context=ctx_str, question=q))
        ]).content
    scores      = judge_answer(q, ctx_str, ans)

    row = {"question": q, "answer_snippet": ans[:120] + "..."}
    for criterion, data in scores.items():
        row[f"{criterion}_score"]  = data.get("score",  "?")
        row[f"{criterion}_reason"] = data.get("reason", "")
    eval_rows.append(row)

df_eval = pd.DataFrame(eval_rows)
print("\n✅ Evaluation complete")

Evaluating: What is the Open Web Index?
Evaluating: How can researchers access European web crawl data?
⚠️  Could not parse judge response as JSON:
 ```json
{
  "faithfulness": {
    "score": 1,
    "reason": "The retrieved context does not provide any information about European web crawl data or how researchers can access it. The answer is entirely unsupported by the context."
  },
  "relevance": {
    "score": 1,
    "reason": "The answer does not address the question at all. The question asks about European web crawl data, but the context retrieved is unrelated to web crawling, data access methods, or European initiatives."
  },
  "completeness": {
    "score": 1,
    "reason": "The answer does not cover any key information available in the context. The context provided is entirely irrelevant to the question posed."
  }
}
```
Evaluating: What is MOSAIC and how does it work?


In [ ]:
# ── 6.5  Display results table 
score_cols = [c for c in df_eval.columns if c.endswith("_score")]

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
display(df_eval[["question"] + score_cols + ["answer_snippet"]])

print("\nMean scores across all questions:")
display(df_eval[score_cols].apply(pd.to_numeric, errors="coerce").mean().rename("mean").to_frame())

---
## What's next?

You've built a complete RAG pipeline! Here are some ideas to extend it:

- **Better chunking:** split long documents into smaller passages before indexing (try different sizes and overlaps in Section 3) and see if retrieval precision improves.
- **Better prompts:** improve the RAG prompt in Section 5 or the judge prompt in Section 6.
- **Agentic experiments:** modify the tool descriptions and the system prompt, and observe how the LLM's source selection changes.
- **Custom evaluation dimensions:** add *conciseness*, *citation quality*, or *tone* to the judge.
- **Different queries:** swap `YOUR_QUESTION` and `EVAL_QUESTIONS` for your own domain or language.

Good luck!